In [1]:
from datasets import load_dataset
import string
from collections import Counter, defaultdict
import math

In [2]:

raw_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

print(f"Train rows: {len(raw_data['train'])}")
print(f"Test rows: {len(raw_data['test'])}")

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Train rows: 36718
Test rows: 4358


In [3]:
def clean_and_tokenize(text):
    text = text.lower()
    
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    tokens = text.split()
    
    if tokens:
        tokens = ['<s>'] + tokens + ['</s>']
        
    return tokens

In [4]:
train_tokens = []
for line in raw_data['train']['text']:
    cleaned = clean_and_tokenize(line)
    if cleaned:
        train_tokens.extend(cleaned)

test_tokens = []
for line in raw_data['test']['text']:
    cleaned = clean_and_tokenize(line)
    if cleaned:
        test_tokens.extend(cleaned)

print(f"Total training tokens: {len(train_tokens)}")
print(f"Total testing tokens: {len(test_tokens)}")

Total training tokens: 1803633
Total testing tokens: 212825


In [5]:
def get_ngrams(tokens, n):
    ngrams = []
    
    for i in range(len(tokens) - n + 1):
        window = tuple(tokens[i : i + n])
        ngrams.append(window)
    return ngrams

unigram_counts = Counter(get_ngrams(train_tokens, 1))
bigram_counts = Counter(get_ngrams(train_tokens, 2))
trigram_counts = Counter(get_ngrams(train_tokens, 3))

vocab = set(train_tokens)
V = len(vocab)

print(f"Unique Unigrams (Vocab Size V): {V}")
print(f"Total Bigrams found: {len(bigram_counts)}")

Unique Unigrams (Vocab Size V): 66174
Total Bigrams found: 705958


In [6]:
def get_ngram_probability(ngram, n, counts, context_counts, vocab_size):
    ngram_count = counts.get(ngram, 0)
    
    if n == 1:
        context_count = sum(counts.values())
    else:
        context = ngram[:-1]
        context_count = context_counts.get(context, 0)
        
    probability = (ngram_count + 1) / (context_count + vocab_size)
    return probability

In [7]:
def calculate_perplexity(test_tokens, n, counts, context_counts, vocab_size):
    test_ngrams = get_ngrams(test_tokens, n)
    N = len(test_ngrams)
    
    log_probability_sum = 0
    
    for ngram in test_ngrams:
        prob = get_ngram_probability(ngram, n, counts, context_counts, vocab_size)
        
        log_probability_sum += math.log2(prob)
        
    avg_log_prob = log_probability_sum / N
    perplexity = 2 ** (-avg_log_prob)
    
    return perplexity

In [8]:
n_values = [1, 2, 3]
results = {}

for n in n_values:
    if n == 1:
        ppl = calculate_perplexity(test_tokens, 1, unigram_counts, None, V)
    elif n == 2:
        ppl = calculate_perplexity(test_tokens, 2, bigram_counts, unigram_counts, V)
    elif n == 3:
        ppl = calculate_perplexity(test_tokens, 3, trigram_counts, bigram_counts, V)
    
    results[n] = ppl
    print(f"{n}-gram Perplexity: {ppl:.2f}")

1-gram Perplexity: 2165.23
2-gram Perplexity: 8404.07
3-gram Perplexity: 39571.97


In [9]:
import pickle

model_params = {
    'unigram_counts': unigram_counts,
    'bigram_counts': bigram_counts,
    'trigram_counts': trigram_counts,
    'vocab': vocab,
    'vocab_size': V
}

with open('ngram_model.pkl', 'wb') as f:
    pickle.dump(model_params, f)

print("Model parameters saved to disk!")

Model parameters saved to disk!


In [10]:
def generate_next_token(context, n, counts, context_counts, vocab, vocab_size):
    best_word = None
    max_prob = -1
    
    for word in vocab:
        ngram = context + (word,)

        prob = get_ngram_probability(ngram, n, counts, context_counts, vocab_size)
        
        if prob > max_prob:
            max_prob = prob
            best_word = word
            
    return best_word

def generate_sequence(prompt, n, max_tokens=30):
    tokens = clean_and_tokenize(prompt)
    
    if tokens[-1] == '</s>': tokens.pop()
    
    for _ in range(max_tokens):
        context = tuple(tokens[-(n-1):])
        
        if n == 2:
            next_word = generate_next_token(context, 2, bigram_counts, unigram_counts, vocab, V)
        elif n == 3:
            next_word = generate_next_token(context, 3, trigram_counts, bigram_counts, vocab, V)
        
        tokens.append(next_word)
        
        if next_word == '</s>':
            break
            
    return " ".join(tokens)

print(generate_sequence("college students", n=3))

<s> college students including some running down the field of view of the season with a large number of other people s party – a kind of a new album would be the
